In [2]:
from _utils import *

def map_subtype(type, subtype):
    if type == 'I Ca':
        if subtype in ['I Ca (R-Type)','I Ca (P/Q-Type)','I Ca (Q-Type)','I Ca (HVA)','I Ca (L-Type HT)','I Ca (N-Type)']:
            return 'I Ca (HVA)'
        elif subtype in ['I Ca (T-type LT)']:
            return 'I Ca (T-type LT)'
        else:
            return 'I Ca (Rare)'
    if type == 'I H':
        return 'I H'
    if type == 'I K':
        if subtype in ['I K (A-Type Transient)','I K (A-Type Slow)']:
            return 'I K (A-type)'
        elif subtype in ['I K (Ca-activated - General - BK, IK, SK, and I AHP currents)','I K (Ca-Activated - Sk)','I K (AHP)','I K (Ca-activated Bk)','I K (C-type)']:
            return 'I K (Ca-activated)'
        elif subtype in ['I K (M-type)','I K (CNQ1)']:
            return 'I K (M-type)'
        elif subtype in ['I K (Delayed Rectifier)']:
            return 'I K (Delayed Rectifier)'
        else:
            return 'I K (Rare)'
    if type == 'I Na':
        if subtype in ['I Na (Transient)','I Na (Persistent)','I Na (Slow inactivation)']:
            return subtype
        else:
            return 'I Na (Rare)'
    if type == 'I Other':
        if subtype == ['I Leak (General)']:
            return 'I Other (Leak)'
        else:
            return 'I Other (Rare)'
    if type == 'Receptor':
        if subtype in ['R Glutamate (AMPA)','R Glutamate (NMDA)']:
            return subtype
        if subtype in ['R GABA (type A)', 'R GABA (type b)','R GABA (general)']:
            return 'R GABA'
        elif subtype in ['R Ion Receptors (General)']:
            return 'R Other (General)'
        else:
            return 'R Other (Rare)' 
    elif subtype in ['Neither - Accumulation Mechanism', 'Neither - Utility','Neither - Clamp','Neither - Localized Reactions','Neither - Artificial Cell','Neither - Gap Junction','Neither - Non-Neural','Neither  - Localized Reactions']:
        return 'Z Neither'
    else:
        return subtype

ModuleNotFoundError: No module named '_utils'

In [1]:
raw_excel_df = pd.read_excel(ANNOTATIONS_FP)

NameError: name 'pd' is not defined

In [3]:
log = DataLogger()
log.add_entry("raw excel file", raw_excel_df)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133


In [4]:
excel2 = raw_excel_df.query("row_id <= 1301")
log.add_entry("annotated rows", excel2)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301


In [5]:
excel3 = excel2.query("type != 'Exclude'")
log.add_entry("drop excluded files", excel3)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226


In [6]:
excel4 = excel3.query("type != 'I Multi'").query("type != 'R Multi'")
log.add_entry("drop multi mechanisms", excel4)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121


In [7]:
excel5 = excel4.query("type != 'I Cl'")
log.add_entry("Remove 1 type Cl", excel5)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120


In [11]:
excel6 = excel5.query("subtype_confidence != '1 - Not confident at all'")
log.add_entry("drop unconfident samples", excel6)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023


In [13]:
TEST_SET = excel5.query("row_id > 1000")["file_hash"].tolist()
excel7 = excel6.query("file_hash not in @TEST_SET")
log.add_entry("remove test set", excel7)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782


In [15]:
SEED = 6
np.random.seed(SEED)

X = excel7.drop(columns=["type", "subtype_1"])
y_type = excel7["type"].astype(str)
y_subtype = excel7["subtype_1"].astype(str)
X_train, X_val, y_type_train, y_type_val, y_subtype_train, y_subtype_val = train_test_split(
    X, y_type, y_subtype, test_size=0.20, stratify=y_type, random_state=SEED
)


TRAIN1_SET = X_train["file_hash"].tolist()
VAL1_SET = X_val["file_hash"].tolist()

In [16]:
excel8 = excel7.query("file_hash in @TRAIN1_SET")
log.add_entry("train set 1 (for labeling only)", excel8)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782
8,train set 1 (for labeling only),625,625


In [17]:
for t, df_sub in excel7.groupby("type"):
    print(f"\n=== {t} ===")
    print(df_sub["subtype_1"].value_counts())
#lump <10 to I Na (Rare)


=== I Ca ===
subtype_1
I Ca (L-Type HT)    26
I Ca (T-type LT)    20
I Ca (HVA)           9
I Ca (R-Type)        7
I Ca (N-Type)        5
I Ca (Ca Pump)       4
I Ca (General)       3
I Ca (P/Q-Type)      3
I Ca (Q-Type)        2
Name: count, dtype: int64

=== I H ===
subtype_1
I H (Hyperpolarization-Activated)    39
Name: count, dtype: int64

=== I K ===
subtype_1
I K (A-Type Transient)                                           73
I K (Delayed Rectifier)                                          58
I K (Ca-activated - General - BK, IK, SK, and I AHP currents)    30
I K (M-type)                                                     29
I K (Ca-Activated - Sk)                                          10
I K (AHP)                                                         9
I K (HT)                                                          6
I K (Ca-activated Bk)                                             5
I K (D-type)                                                      5
I K (IR)           

In [18]:
excel8 = excel8.copy()
excel8["label"] = excel8.apply(
    lambda row: map_subtype(row["type"], row["subtype_1"]), axis=1
)


In [19]:
excel8[["label","subtype_1"]].groupby(["label","subtype_1"]).count()

Empty DataFrame
Columns: []
Index: [(I Ca (HVA), I Ca (HVA)), (I Ca (HVA), I Ca (L-Type HT)), (I Ca (HVA), I Ca (N-Type)), (I Ca (HVA), I Ca (P/Q-Type)), (I Ca (HVA), I Ca (Q-Type)), (I Ca (HVA), I Ca (R-Type)), (I Ca (Rare), I Ca (Ca Pump)), (I Ca (Rare), I Ca (General)), (I Ca (T-type LT), I Ca (T-type LT)), (I H, I H (Hyperpolarization-Activated)), (I K (A-type), I K (A-Type Slow)), (I K (A-type), I K (A-Type Transient)), (I K (Ca-activated), I K (AHP)), (I K (Ca-activated), I K (C-type)), (I K (Ca-activated), I K (Ca-Activated - Sk)), (I K (Ca-activated), I K (Ca-activated - General - BK, IK, SK, and I AHP currents)), (I K (Ca-activated), I K (Ca-activated Bk)), (I K (Delayed Rectifier), I K (Delayed Rectifier)), (I K (M-type), I K (CNQ1)), (I K (M-type), I K (M-type)), (I K (Rare), I K (D-type)), (I K (Rare), I K (HT)), (I K (Rare), I K (IR)), (I K (Rare), I K (IR, inactivating)), (I K (Rare), I K (LT)), (I K (Rare), I K (Leak)), (I K (Rare), I K (Slow)), (I K (Rare), I K (Ultra Rapid Shaker-related Cardiac)), (I K (Rare), I Kx (Photoreceptor)), (I Na (Persistent), I Na (Persistent)), (I Na (Rare), I Na (General)), (I Na (Rare), I Na (Leak)), (I Na (Rare), I Na (Resurgent)), (I Na (Slow inactivation), I Na (Slow inactivation)), (I Na (Transient), I Na (Transient)), (I Other (Rare), I (Nonspecific Ca-Activated)), (I Other (Rare), I Leak (General)), (I Other (Rare), I Other (CNG)), (I Other (Rare), I Other (Channelrhodopsin)), (I Other (Rare), I Other (KCC2 transporter)), (I Other (Rare), I Other (Modulator-Activated)), (I Other (Rare), I Other (Na/Ca exchanger)), (I Other (Rare), I Other (Na/K pump)), (R GABA, R GABA (general)), (R GABA, R GABA (type A)), (R GABA, R GABA (type b)), (R Glutamate (AMPA), R Glutamate (AMPA)), (R Glutamate (NMDA), R Glutamate (NMDA)), (Z Neither, Neither  - Localized Reactions), (Z Neither, Neither - Accumulation Mechanism), (Z Neither, Neither - Artificial Cell), (Z Neither, Neither - Clamp), (Z Neither, Neither - Gap Junction), (Z Neither, Neither - Localized Reactions), (Z Neither, Neither - Non-Neural), (Z Neither, Neither - Utility)]

In [20]:
excel9 = excel6.copy()
excel9["label"] = excel8.apply(
    lambda row: map_subtype(row["type"], row["subtype_1"]), axis=1
)
log.add_entry("adding labels to the entire dataset", excel9)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782
8,train set 1 (for labeling only),625,625
9,adding labels to the entire dataset,1023,1023


In [21]:
excel10 = excel9.query("file_hash not in @TEST_SET")
log.add_entry("remove test set", excel10)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782
8,train set 1 (for labeling only),625,625
9,adding labels to the entire dataset,1023,1023


In [22]:
SEED = 6
np.random.seed(SEED)

X = excel10.drop(columns=["type", "subtype_1","label"])
y_type = excel10["type"].astype(str)
y_subtype = excel10["label"].astype(str)
X_train, X_val, y_type_train, y_type_val, y_subtype_train, y_subtype_val = train_test_split(
    X, y_type, y_subtype, test_size=0.20, stratify=y_type, random_state=SEED
)

TRAIN2_SET = X_train["file_hash"].tolist()
VAL2_SET = X_val["file_hash"].tolist()

In [23]:
excel11 = excel10.query("file_hash in @TRAIN2_SET")
log.add_entry("training set (actual)", excel11)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782
8,train set 1 (for labeling only),625,625
9,adding labels to the entire dataset,1023,1023


In [24]:
excel12 = excel10.query("file_hash in @VAL2_SET")
log.add_entry("val set (actual)", excel12)
log.get_log()

,step,n_row,n_hash
0,raw excel file,5133,5133
1,annotated rows,1301,1301
2,drop excluded files,1226,1226
3,drop multi mechanisms,1121,1121
4,Remove 1 type Cl,1120,1120
5,drop unconfident samples,1023,1023
6,remove test set,1023,1023
7,remove test set,782,782
8,train set 1 (for labeling only),625,625
9,adding labels to the entire dataset,1023,1023


In [25]:
df_train = pd.DataFrame({"file_hash": TRAIN2_SET, "set": "train"})
df_test  = pd.DataFrame({"file_hash": TEST_SET, "set": "test"})
df_val   = pd.DataFrame({"file_hash": VAL2_SET, "set": "val"})

In [26]:
df_split = pd.concat([df_train, df_test, df_val], ignore_index=True)

In [21]:
excel8.columns

Index(['row_id', 'file_hash', 'raw_sha', 'count', 'url', 'mod_file', 'type',
       'subtype_1', 'subtype_2', 'subtype_3', 'subtype_4', 'subtype_5',
       'subtype_confidence', 'ask_robert', 'annotated', 'Robert reviewed',
       'notes_free_text', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19',
       'Unnamed: 20', 'label'],
      dtype='object')

In [27]:
keep_cols = ["row_id","file_hash","url","type","label","subtype_1","subtype_confidence","Robert reviewed","notes_free_text"]
excel13 = excel9[keep_cols]

In [28]:
excel14 = excel13.merge(df_split, on="file_hash")


In [30]:
table1 = pd.crosstab(excel13["label"], excel14["set"])

table1

set,test,train,val
label,,,
I Ca (HVA),10,26,4
I Ca (Rare),1,5,0
I Ca (T-type LT),5,9,3
I H,8,16,7
I K (A-type),14,43,5
I K (Ca-activated),9,25,8
I K (Delayed Rectifier),7,30,8
I K (M-type),4,17,4
I K (Rare),8,16,5


In [31]:
table1_pct = pd.crosstab(excel13["label"], excel14["set"], normalize="index") * 100

table1_pct

set,test,train,val
label,,,
I Ca (HVA),25.000000,65.000000,10.000000
I Ca (Rare),16.666667,83.333333,0.000000
I Ca (T-type LT),29.411765,52.941176,17.647059
I H,25.806452,51.612903,22.580645
I K (A-type),22.580645,69.354839,8.064516
I K (Ca-activated),21.428571,59.523810,19.047619
I K (Delayed Rectifier),15.555556,66.666667,17.777778
I K (M-type),16.000000,68.000000,16.000000
I K (Rare),27.586207,55.172414,17.241379


In [32]:
labels = (
    excel14["label"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)


In [33]:
labels

0                   I Ca (HVA)
1                  I Ca (Rare)
2             I Ca (T-type LT)
3                          I H
4                 I K (A-type)
5           I K (Ca-activated)
6      I K (Delayed Rectifier)
7                 I K (M-type)
8                   I K (Rare)
9            I Na (Persistent)
10                 I Na (Rare)
11    I Na (Slow inactivation)
12            I Na (Transient)
13              I Other (Rare)
14                      R GABA
15          R Glutamate (AMPA)
16          R Glutamate (NMDA)
17                   Z Neither
Name: label, dtype: object

In [34]:
excel14.to_csv(PIPELINE_DATA_DIR  /"anno_with_split.csv", index=False)
df_split.to_csv(PIPELINE_DATA_DIR /"split.csv", index=False)
labels.to_csv(PIPELINE_DATA_DIR / "labels.csv", index=False)